In [8]:
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import pandas as pd

In [9]:
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BartForConditionalGeneration(
  (model): BartModel(
    (shared): Embedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): Embedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_norm): La

In [10]:
def get_rankings_for_notebook(df, min_reviews=3):
    # Standardize scoring
    if "rating" in df.columns:
        score_col = "rating"
        neutral = 3.0
    else:
        sentiment_map = {"Positive": 1, "Neutral": 0, "Negative": -1}
        df["temp_score"] = df["sentiment"].map(sentiment_map)
        score_col = "temp_score"
        neutral = 0

    rankings = {}
    for cat in df["category"].unique():
        cat_df = df[df["category"] == cat]
        stats = cat_df.groupby("product_name")[score_col].agg(["mean", "count"])
        qualified = stats[stats["count"] >= min_reviews]

        best = qualified[qualified["mean"] > neutral].nlargest(3, "mean").index.tolist()
        worst = qualified[qualified["mean"] <= neutral].nsmallest(1, "mean").index.tolist()

        rankings[cat] = {"best": best, "worst": worst}
    return rankings

In [11]:
def generate_summary(product_name, reviews, category, is_best=True):
    if not reviews or len(reviews.strip()) < 20:
        return "Consistent quality across all recorded customer interactions."

    # Add instructions for the model
    role = "top-rated highlight" if is_best else "critical warning"
    prefix = f"Summarize why '{product_name}' is a {role} in the {category} category: "

    input_text = prefix + reviews

    # Tuning the generation parameters
    inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=80,
        min_length=30,
        do_sample=True,     # <--- Change to True for more natural phrasing
        top_k=50,           # <--- Helps prevent repetitive/clumsy loops
        top_p=0.92,
        repetition_penalty=2.5, # <--- Forces the model to stop repeating rants
        length_penalty=1.5,
        num_beams=5
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [13]:
# Load the data
df = pd.read_csv("results_test.csv")

# Get product targets
all_rankings = get_rankings_for_notebook(df)

# Generate summaries
for category, items in all_rankings.items():
    print(f"\n{'='*30}")
    print(f"MARKET REPORT: {category.upper()}")
    print(f"{'='*30}\n")

    print("### TOP PERFORMING PRODUCTS\n")
    for p_name in items["best"]:
        # Gather all reviews for this specific product
        reviews_blob = " ".join(df[df["product_name"] == p_name]["review_text"].astype(str))
        summary = generate_summary(p_name, reviews_blob, category, is_best=True)
        print(f"**{p_name}**")
        print(f"Insight: {summary}\n")

    print("### PRODUCT TO AVOID\n")
    if items["worst"]:
        w_name = items["worst"][0]
        w_reviews_blob = " ".join(df[df["product_name"] == w_name]["review_text"].astype(str))
        w_summary = generate_summary(w_reviews_blob, max_len=60)
        print(f"**{w_name}**")
        print(f"Warning: {w_summary}\n")
    else:
        print("No low-performing products identified in this category.")


MARKET REPORT: SMART SPEAKER

### TOP PERFORMING PRODUCTS

**Echo Dot**
Insight: Decent speaker, but Alexa is annoying. Great for setting timers in the kitchen. Dropped it and it still works. Music quality is lacking base.

### PRODUCT TO AVOID

No low-performing products identified in this category.

MARKET REPORT: HOME APPLIANCES

### TOP PERFORMING PRODUCTS

**Air Fryer**
Insight: Good value, but the basket is a bit small. Hard to clean the grease out of the top. Food comes out dry if you aren't careful. Takes up way too much counter space.

**Kindle Paperwhite**
Insight: Best purchase I've made for travel. Screen is clear even in bright sunlight. Battery life is not as long as advertised. Updating the software took forever. Waterproof feature actually works! A bit slow to respond to touch. Best way to store 1000s of books.

**French Press**
Insight: Coffee is lukewarm by the second cup. Glass broke during the first wash. The plunger gets stuck sometimes. Good for loose leaf tea as